In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
def clean_text(text):
    """
    Clean requirement text:
    - lowercase
    - remove special characters
    - remove extra spaces
    """
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def load_and_preprocess_data(
    csv_path="/content/drive/MyDrive/SoftwareEng/Pure Ds/Pure_Annotate_Dataset.csv",
    test_size=0.2,
    random_state=42
):
    # Load dataset
    df = pd.read_csv(csv_path, encoding="latin1")

    # Keep required columns only
    df = df[["sentence", "NFR_boolean"]]

    # Drop missing values
    df.dropna(inplace=True)

    # Clean text
    df["sentence"] = df["sentence"].apply(clean_text)

    # Encode labels
    # Functional -> 0, Non-Functional -> 1
    # Corrected: Directly use NFR_boolean values as they are already 0 or 1
    df["label"] = df["NFR_boolean"]

    X = df["sentence"]
    y = df["label"]

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    # TF-IDF Vectorization
    vectorizer = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        stop_words="english"
    )

    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    return (
        X_train_tfidf,
        X_test_tfidf,
        y_train,
        y_test,
        vectorizer
    )

Logistic Reg

In [ ]:
X_train_tfidf, X_test_tfidf, y_train, y_test, vectorizer = load_and_preprocess_data()
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",  # important for imbalanced FR/NFR
    random_state=42
)

lr_model.fit(X_train_tfidf, y_train)
y_pred = lr_model.predict(X_test_tfidf)
y_prob = lr_model.predict_proba(X_test_tfidf)[:, 1]
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

Accuracy  : 0.8837
Precision : 0.6146
Recall    : 0.8141
F1-score  : 0.7005


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Functional", "Non-Functional"]
))

                precision    recall  f1-score   support

    Functional       0.96      0.90      0.93      1906
Non-Functional       0.61      0.81      0.70       382

      accuracy                           0.88      2288
     macro avg       0.79      0.86      0.81      2288
  weighted avg       0.90      0.88      0.89      2288



In [ ]:
lr_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}

lr_results

{'Model': 'Logistic Regression',
 'Accuracy': 0.8837412587412588,
 'Precision': 0.6146245059288538,
 'Recall': 0.8141361256544503,
 'F1-score': 0.7004504504504504}

Navie Bayes

In [ ]:
X_train_tfidf, X_test_tfidf, y_train, y_test, vectorizer = load_and_preprocess_data()
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)
y_prob_nb = nb_model.predict_proba(X_test_tfidf)[:, 1]
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)

print(f"Accuracy  : {accuracy_nb:.4f}")
print(f"Precision : {precision_nb:.4f}")
print(f"Recall    : {recall_nb:.4f}")
print(f"F1-score  : {f1_nb:.4f}")

Accuracy  : 0.8942
Precision : 0.8211
Recall    : 0.4686
F1-score  : 0.5967


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_nb,
    target_names=["Functional", "Non-Functional"]
))

                precision    recall  f1-score   support

    Functional       0.90      0.98      0.94      1906
Non-Functional       0.82      0.47      0.60       382

      accuracy                           0.89      2288
     macro avg       0.86      0.72      0.77      2288
  weighted avg       0.89      0.89      0.88      2288



In [ ]:
nb_results = {
    "Model": "Naive Bayes",
    "Accuracy": accuracy_nb,
    "Precision": precision_nb,
    "Recall": recall_nb,
    "F1-score": f1_nb
}

nb_results

{'Model': 'Naive Bayes',
 'Accuracy': 0.8942307692307693,
 'Precision': 0.8211009174311926,
 'Recall': 0.468586387434555,
 'F1-score': 0.5966666666666667}

SVM

In [ ]:
X_train_tfidf, X_test_tfidf, y_train, y_test, vectorizer = load_and_preprocess_data()
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)


svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print(f"Accuracy  : {accuracy_svm:.4f}")
print(f"Precision : {precision_svm:.4f}")
print(f"Recall    : {recall_svm:.4f}")
print(f"F1-score  : {f1_svm:.4f}")

Accuracy  : 0.8973
Precision : 0.6623
Recall    : 0.7853
F1-score  : 0.7186


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Functional", "Non-Functional"]
))

                precision    recall  f1-score   support

    Functional       0.96      0.92      0.94      1906
Non-Functional       0.66      0.79      0.72       382

      accuracy                           0.90      2288
     macro avg       0.81      0.85      0.83      2288
  weighted avg       0.91      0.90      0.90      2288



In [ ]:
svm_results = {
    "Model": "SVM (Linear)",
    "Accuracy": accuracy_svm,
    "Precision": precision_svm,
    "Recall": recall_svm,
    "F1-score": f1_svm
}

svm_results

{'Model': 'SVM (Linear)',
 'Accuracy': 0.8972902097902098,
 'Precision': 0.6622516556291391,
 'Recall': 0.7853403141361257,
 'F1-score': 0.718562874251497}

Random Forest

In [ ]:
X_train_tfidf, X_test_tfidf, y_train, y_test, vectorizer = load_and_preprocess_data()
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_tfidf, y_train)
y_pred_rf = rf_model.predict(X_test_tfidf)
y_prob_rf = rf_model.predict_proba(X_test_tfidf)[:, 1]
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print(f"Accuracy  : {accuracy_rf:.4f}")
print(f"Precision : {precision_rf:.4f}")
print(f"Recall    : {recall_rf:.4f}")
print(f"F1-score  : {f1_rf:.4f}")

Accuracy  : 0.9161
Precision : 0.8958
Recall    : 0.5628
F1-score  : 0.6913


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_rf,
    target_names=["Functional", "Non-Functional"]
))

                precision    recall  f1-score   support

    Functional       0.92      0.99      0.95      1906
Non-Functional       0.90      0.56      0.69       382

      accuracy                           0.92      2288
     macro avg       0.91      0.77      0.82      2288
  weighted avg       0.91      0.92      0.91      2288



In [ ]:
rf_results = {
    "Model": "Random Forest",
    "Accuracy": accuracy_rf,
    "Precision": precision_rf,
    "Recall": recall_rf,
    "F1-score": f1_rf
}

rf_results

{'Model': 'Random Forest',
 'Accuracy': 0.916083916083916,
 'Precision': 0.8958333333333334,
 'Recall': 0.56282722513089,
 'F1-score': 0.6913183279742765}

Gradient Boosting

In [ ]:
X_train_tfidf, X_test_tfidf, y_train, y_test, vectorizer = load_and_preprocess_data()
from sklearn.ensemble import GradientBoostingClassifier

# Convert sparse to dense
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

gb_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train_dense, y_train)
y_pred_gb = gb_model.predict(X_test_dense)
y_prob_gb = gb_model.predict_proba(X_test_dense)[:, 1]
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_gb = accuracy_score(y_test, y_pred_gb)
precision_gb = precision_score(y_test, y_pred_gb)
recall_gb = recall_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb)

print(f"Accuracy  : {accuracy_gb:.4f}")
print(f"Precision : {precision_gb:.4f}")
print(f"Recall    : {recall_gb:.4f}")
print(f"F1-score  : {f1_gb:.4f}")

Accuracy  : 0.8890
Precision : 0.8516
Recall    : 0.4058
F1-score  : 0.5496


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_gb,
    target_names=["Functional", "Non-Functional"]
))

                precision    recall  f1-score   support

    Functional       0.89      0.99      0.94      1906
Non-Functional       0.85      0.41      0.55       382

      accuracy                           0.89      2288
     macro avg       0.87      0.70      0.74      2288
  weighted avg       0.89      0.89      0.87      2288



In [ ]:
gb_results = {
    "Model": "Gradient Boosting",
    "Accuracy": accuracy_gb,
    "Precision": precision_gb,
    "Recall": recall_gb,
    "F1-score": f1_gb
}

gb_results

{'Model': 'Gradient Boosting',
 'Accuracy': 0.888986013986014,
 'Precision': 0.8516483516483516,
 'Recall': 0.40575916230366493,
 'F1-score': 0.549645390070922}

Comparision

In [ ]:
import pandas as pd

results_df = pd.DataFrame([
    lr_results,
    nb_results,
    svm_results,
    rf_results,
    gb_results
])

results_df

,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.883741,0.614625,0.814136,0.700450
1,Naive Bayes,0.894231,0.821101,0.468586,0.596667
2,SVM (Linear),0.897290,0.662252,0.785340,0.718563
3,Random Forest,0.916084,0.895833,0.562827,0.691318
4,Gradient Boosting,0.888986,0.851648,0.405759,0.549645


In [ ]:
results_df_sorted = results_df.sort_values(
    by="F1-score",
    ascending=False
)

results_df_sorted

,Model,Accuracy,Precision,Recall,F1-score
2,SVM (Linear),0.897290,0.662252,0.785340,0.718563
0,Logistic Regression,0.883741,0.614625,0.814136,0.700450
3,Random Forest,0.916084,0.895833,0.562827,0.691318
1,Naive Bayes,0.894231,0.821101,0.468586,0.596667
4,Gradient Boosting,0.888986,0.851648,0.405759,0.549645


In [ ]:
best_model = results_df_sorted.iloc[0]
best_model

,2
Model,SVM (Linear)
Accuracy,0.89729
Precision,0.662252
Recall,0.78534
F1-score,0.718563


In [ ]:
from sklearn.svm import LinearSVC

final_svm = LinearSVC(
    class_weight="balanced",
    random_state=42,
    max_iter=5000
)

final_svm.fit(X_train_tfidf, y_train)

LinearSVC(class_weight='balanced', max_iter=5000, random_state=42)

In [ ]:
import joblib

# Save model
joblib.dump(final_svm, "svm_final_model.pkl")

# Save TF-IDF vectorizer
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully")

Model and vectorizer saved successfully


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

joblib.dump(final_svm, "/content/drive/MyDrive/SoftwareEng/svm_final_model.pkl")
joblib.dump(vectorizer, "/content/drive/MyDrive/SoftwareEng/tfidf_vectorizer.pkl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['/content/drive/MyDrive/SoftwareEng/tfidf_vectorizer.pkl']